In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# eval_v9.py — Standalone Evaluation Pipeline for V9 DualEncoderUNet
# ═══════════════════════════════════════════════════════════════════════════════
#
# PURPOSE:
#   Load a saved V9 checkpoint and run a full evaluation on val and test splits.
#   Supports three modes:
#     1. Standard eval (fast)
#     2. TTA eval — 4-flip test-time augmentation (more accurate)
#     3. Threshold sweep on val (find the best operating threshold)
#
# USAGE (Kaggle notebook cell):
#   exec(open("eval_v9.py").read())
#
# OR paste directly into a notebook cell and run.
# ═══════════════════════════════════════════════════════════════════════════════
 
import os
import random
import numpy as np
import torch
import torch.nn as nn
import tifffile
import albumentations as A
 
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision.models import ResNet34_Weights
import torchvision.models as tvm
 
# ──────────────────────────────────────────────────────────────────────────────
# SECTION 1 — CONFIG
# Edit these two lines to point at your checkpoint and dataset.
# Everything else auto-derives from them.
# ──────────────────────────────────────────────────────────────────────────────
 
CHECKPOINT = "/kaggle/input/models/rohitrobert34/v9/pytorch/default/1/best_v9.pth"   # filename of the saved .pth weights
 
# Dataset root — V9 was trained on the v1 path (rohitrobert34, not rohitrobertc2)
# If your checkpoint was trained on v2, change this to:
#   "/kaggle/input/datasets/rohitrobertc2/galaxyeye-change-detection-v2"
BASE_DIR = Path("/kaggle/input/datasets/rohitrobert34/galaxyeye-change-detection")
 
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# SECTION 2 — CONSTANTS (match V9 training config exactly)
# ──────────────────────────────────────────────────────────────────────────────
 
SEED        = 42
IN_CH_EO    = 3    # EO encoder input channels (RGB)
IN_CH_SAR   = 6    # SAR encoder input channels (post, eo_lum, cross, cross_abs, ndvi, sar_ndvi)
NUM_CLASSES = 1
BATCH_SIZE  = 4
NUM_WORKERS = 2
CROP_SIZE   = 512
THRESHOLD   = 0.4  # default prediction threshold (sigmoid output > THRESHOLD → change)
 
# Loss params (must match training — used to compute eval loss)
FOCAL_ALPHA  = 0.2
FOCAL_BETA   = 0.8
FOCAL_GAMMA  = 1.5   # CORRECTED value (was 0.75 in V1-V4)
FOCAL_SMOOTH = 1.0
 
# Threshold sweep range — applied to val only to find best threshold
SWEEP_THRESHOLDS = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
 
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# SECTION 3 — REPRODUCIBILITY + DEVICE
# ──────────────────────────────────────────────────────────────────────────────
 
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
 

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# SECTION 4 — PATHS + FILE DISCOVERY
# ──────────────────────────────────────────────────────────────────────────────
 
PATHS = {
    "val": {
        "pre":    BASE_DIR / "change_detection_assignment_data" / "val" / "pre-event",
        "post":   BASE_DIR / "change_detection_assignment_data" / "val" / "post-event",
        "target": BASE_DIR / "change_detection_assignment_data" / "val" / "target",
    },
    "test": {
        "pre":    BASE_DIR / "change_detection_assignment_data" / "test" / "pre-event",
        "post":   BASE_DIR / "change_detection_assignment_data" / "test" / "post-event",
        "target": BASE_DIR / "change_detection_assignment_data" / "test" / "target",
    },
}
 
FILES = {}
for split in ["val", "test"]:
    FILES[split] = sorted([p.name for p in PATHS[split]["pre"].glob("*.tif")])
 
for split in ["val", "test"]:
    n = len(FILES[split])
    print(f"{split}: {n} files | first: {FILES[split][0]}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# SECTION 5 — DATASET
# Exact copy of DualChangeDataset from V9 — DO NOT modify feature engineering.
# The model weights were baked with these exact channel definitions.
# ──────────────────────────────────────────────────────────────────────────────
 
class DualChangeDataset(Dataset):
    """
    Returns (eo_tensor, sar_tensor, mask_tensor, fname) per sample.
 
    EO tensor  : (3, H, W) float32 — normalised pre-event RGB
    SAR tensor : (6, H, W) float32 — [post_norm, eo_lum, cross, cross_abs, ndvi, sar_ndvi]
    mask tensor: (H, W)    float32 — binary {0=no change, 1=change}
    """
 
    def __init__(self, split, paths, file_names, transform=None):
        self.split      = split
        self.paths      = paths
        self.file_names = file_names
        self.transform  = transform
 
    def __len__(self):
        return len(self.file_names)
 
    def _load_triplet(self, idx):
        fname = self.file_names[idx]
 
        pre  = tifffile.imread(self.paths[self.split]["pre"]    / fname)  # (H,W,3) uint8
        post = tifffile.imread(self.paths[self.split]["post"]   / fname)  # (H,W)   uint8
        mask = tifffile.imread(self.paths[self.split]["target"] / fname)  # (H,W)   uint8
 
        # SAR: add channel dimension
        post = post[:, :, np.newaxis]                                      # (H,W,1)
 
        # CRITICAL: remap mask — 0=background, 1=uncertain, 2+=change
        # V9 has smart remapping: uses >=1 if no >=2 values exist (scene_09 fix)
        unique_vals = np.unique(mask)
        if np.any(unique_vals >= 2):
            mask = np.where(mask >= 2, 1, 0).astype(np.uint8)
        else:
            mask = np.where(mask >= 1, 1, 0).astype(np.uint8)
 
        return pre, post, mask, fname
 
    def _engineer_features(self, pre, post):
        # Step 1: Normalise (ALWAYS before computing derived features)
        pre_norm  = pre.astype(np.float32) / 255.0              # (H,W,3) [0,1]
        post_norm = (post.astype(np.float32) - 1.0) / 230.0     # (H,W,1) [0,1]
        post_norm = np.clip(post_norm, 0.0, 1.0)
 
        # Step 2: EO luminance — weighted brightness of pre-event image
        eo_lum = (
            0.299 * pre_norm[:, :, 0:1] +   # Red
            0.587 * pre_norm[:, :, 1:2] +   # Green
            0.114 * pre_norm[:, :, 2:3]     # Blue
        )                                                        # (H,W,1) [0,1]
 
        # Step 3: Cross-modal disagreement
        # Positive → SAR bright, EO dim → rubble
        # Negative → SAR dark, EO medium → flood
        # Near zero → sensors agree → no change
        cross     = post_norm - eo_lum                           # (H,W,1) [-1,1]
        cross_abs = np.abs(cross)                                # (H,W,1) [0,1]
 
        # Step 4: NDVI (green-red difference) — vegetation damage indicator
        ndvi = (
            (pre_norm[:, :, 1:2] - pre_norm[:, :, 0:1]) /
            (pre_norm[:, :, 1:2] + pre_norm[:, :, 0:1] + 1e-8)
        )                                                        # (H,W,1) [-1,1]
 
        # Step 5: SAR-NDVI interaction — combined rubble + vegetation loss
        sar_ndvi = post_norm * (1.0 - np.clip(ndvi, 0.0, 1.0))  # (H,W,1) [0,1]
 
        # Split into EO and SAR stacks for separate encoders
        eo_stack  = pre_norm                                     # (H,W,3) → ResNet34
        sar_stack = np.concatenate(
            [post_norm, eo_lum, cross, cross_abs, ndvi, sar_ndvi],
            axis=-1
        ).astype(np.float32)                                     # (H,W,6) → ResNet34 (patched)
 
        return eo_stack, sar_stack
 
    def __getitem__(self, idx):
        pre, post, mask, fname = self._load_triplet(idx)
        eo_stack, sar_stack   = self._engineer_features(pre, post)
 
        if self.transform is not None:
            # additional_targets ensures SAR gets IDENTICAL spatial transform as EO
            augmented = self.transform(
                image=eo_stack,
                image2=sar_stack,
                mask=mask
            )
            eo_stack  = augmented["image"]
            sar_stack = augmented["image2"]
            mask      = augmented["mask"]
 
        eo_tensor   = torch.from_numpy(np.transpose(eo_stack,  (2, 0, 1))).float()  # (3,H,W)
        sar_tensor  = torch.from_numpy(np.transpose(sar_stack, (2, 0, 1))).float()  # (6,H,W)
        mask_tensor = torch.from_numpy(mask).float()                                 # (H,W)
 
        return eo_tensor, sar_tensor, mask_tensor, fname
 
 
# Val and test: centre crop only (no augmentation — we want deterministic results)
val_test_transform = A.Compose([
    A.CenterCrop(CROP_SIZE, CROP_SIZE, p=1.0),
], additional_targets={"image2": "image"},   # sync EO and SAR spatial transform
   is_check_shapes=False)
 
# Build datasets
val_dataset  = DualChangeDataset("val",  PATHS, FILES["val"],  val_test_transform)
test_dataset = DualChangeDataset("test", PATHS, FILES["test"], val_test_transform)
 
print(f"\nDatasets ready — Val: {len(val_dataset)} | Test: {len(test_dataset)}")
 
# Build loaders
val_loader = DataLoader(
    val_dataset,  batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
 
print(f"Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")
 

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# SECTION 6 — MODEL ARCHITECTURE (exact copy from V9)
# ──────────────────────────────────────────────────────────────────────────────
 
class ChannelAttention(nn.Module):
    """Squeeze-and-excite style: avg+max pool → MLP → sigmoid gate."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        mid = max(channels // reduction, 1)
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()
 
    def forward(self, x):
        B, C, H, W = x.shape
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        gate    = self.sigmoid(avg_out + max_out).view(B, C, 1, 1)
        return x * gate
 
 
class SpatialAttention(nn.Module):
    """Compress channels → conv → sigmoid gate over spatial locations."""
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size=kernel_size,
                                 padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
 
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        gate = self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))
        return x * gate
 
 
class CBAM(nn.Module):
    """Full CBAM: channel attention → spatial attention."""
    def __init__(self, channels, reduction=16, spatial_kernel=7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(spatial_kernel)
 
    def forward(self, x):
        return self.sa(self.ca(x))
 
 
class ConvBlock(nn.Module):
    """Two 3×3 conv → BN → ReLU."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
 
    def forward(self, x):
        return self.block(x)
 
 
class FusionUpBlock(nn.Module):
    """
    Decoder stage: upsample → concat(eo_skip, sar_skip) → ConvBlock → optional CBAM.
    Takes features from BOTH encoders as skip connections.
    """
    def __init__(self, in_ch, eo_skip_ch, sar_skip_ch, out_ch, use_cbam=False):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = ConvBlock(in_ch // 2 + eo_skip_ch + sar_skip_ch, out_ch)
        self.cbam = CBAM(out_ch) if use_cbam else nn.Identity()
 
    def forward(self, x, eo_skip, sar_skip):
        x = self.up(x)
        x = torch.cat([x, eo_skip, sar_skip], dim=1)
        return self.cbam(self.conv(x))
 
 
class DualEncoderUNet(nn.Module):
    """
    V9 Architecture:
      EO  branch: ResNet34 (3ch) → e0..e4
      SAR branch: ResNet34 patched (6ch) → s0..s4
      Bottleneck: cat(eo_b, sar_b) → CBAM(1024ch)
      Decoder: FusionUpBlock × 4, CBAM on d3 + d2
      Head: ConvTranspose → Conv(1) → raw logits
    """
 
    def __init__(self, in_ch_eo=IN_CH_EO, in_ch_sar=IN_CH_SAR, num_classes=NUM_CLASSES):
        super().__init__()
 
        # ── EO Encoder (ResNet34, standard 3-channel input) ───────────────────
        eo_enc       = tvm.resnet34(weights=ResNet34_Weights.DEFAULT)
        self.eo_e0   = nn.Sequential(eo_enc.conv1, eo_enc.bn1, eo_enc.relu)
        self.eo_pool = eo_enc.maxpool
        self.eo_e1   = eo_enc.layer1   # out: (B, 64,  H/4,  W/4)
        self.eo_e2   = eo_enc.layer2   # out: (B, 128, H/8,  W/8)
        self.eo_e3   = eo_enc.layer3   # out: (B, 256, H/16, W/16)
        self.eo_e4   = eo_enc.layer4   # out: (B, 512, H/32, W/32)
 
        # ── SAR Encoder (ResNet34 patched for in_ch_sar channels) ─────────────
        sar_enc  = tvm.resnet34(weights=ResNet34_Weights.DEFAULT)
        old_w    = sar_enc.conv1.weight.data              # (64, 3, 7, 7)
        new_conv = nn.Conv2d(in_ch_sar, 64, kernel_size=7,
                             stride=2, padding=3, bias=False)
        with torch.no_grad():
            repeated = old_w.repeat(1, in_ch_sar // 3 + 1, 1, 1)[:, :in_ch_sar]
            new_conv.weight.data = repeated / (in_ch_sar / 3)
        sar_enc.conv1   = new_conv
        self.sar_e0     = nn.Sequential(sar_enc.conv1, sar_enc.bn1, sar_enc.relu)
        self.sar_pool   = sar_enc.maxpool
        self.sar_e1     = sar_enc.layer1
        self.sar_e2     = sar_enc.layer2
        self.sar_e3     = sar_enc.layer3
        self.sar_e4     = sar_enc.layer4
 
        # ── Bottleneck CBAM (1024 = 512 EO + 512 SAR) ────────────────────────
        self.bottleneck_cbam = CBAM(1024)
 
        # ── Decoder ───────────────────────────────────────────────────────────
        self.d3 = FusionUpBlock(1024, 256, 256, 256, use_cbam=True)
        self.d2 = FusionUpBlock( 256, 128, 128, 128, use_cbam=True)
        self.d1 = FusionUpBlock( 128,  64,  64,  64, use_cbam=False)
        self.d0 = FusionUpBlock(  64,  64,  64,  32, use_cbam=False)
 
        self.up_final = nn.ConvTranspose2d(32, 32, kernel_size=2, stride=2)
        self.head     = nn.Conv2d(32, num_classes, kernel_size=1)
 
    def forward(self, eo, sar):
        # EO path
        eo_s0 = self.eo_e0(eo)
        eo_s1 = self.eo_e1(self.eo_pool(eo_s0))
        eo_s2 = self.eo_e2(eo_s1)
        eo_s3 = self.eo_e3(eo_s2)
        eo_b  = self.eo_e4(eo_s3)
 
        # SAR path
        sar_s0 = self.sar_e0(sar)
        sar_s1 = self.sar_e1(self.sar_pool(sar_s0))
        sar_s2 = self.sar_e2(sar_s1)
        sar_s3 = self.sar_e3(sar_s2)
        sar_b  = self.sar_e4(sar_s3)
 
        # Bottleneck fusion
        bottleneck = self.bottleneck_cbam(torch.cat([eo_b, sar_b], dim=1))
 
        # Decoder (each stage fuses both encoder skip connections)
        x = self.d3(bottleneck, eo_s3, sar_s3)
        x = self.d2(x,          eo_s2, sar_s2)
        x = self.d1(x,          eo_s1, sar_s1)
        x = self.d0(x,          eo_s0, sar_s0)
 
        x = self.up_final(x)
        return self.head(x)   # raw logits: (B, 1, H, W)
 

In [ ]:
import torch
import torch.nn as nn

class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.2, beta=0.8, gamma=1.5, smooth=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, pred, target):
        target = target.to(pred.device)
        pred = torch.sigmoid(pred).view(-1)
        target = target.view(-1).float()

        tp = (pred * target).sum()
        fp = ((1 - target) * pred).sum()
        fn = (target * (1 - pred)).sum()

        tversky = (tp + self.smooth) / (
            tp + self.alpha * fp + self.beta * fn + self.smooth
        )
        return (1 - tversky) ** self.gamma


class FocalBCELoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, pred, target):
        target = target.to(pred.device)
        pred = torch.sigmoid(pred).view(-1)
        target = target.view(-1).float()

        eps = 1e-8
        p_t = pred * target + (1 - pred) * (1 - target)
        alpha_t = self.alpha * target + (1 - self.alpha) * (1 - target)
        focal_weight = alpha_t * (1 - p_t).pow(self.gamma)

        bce = -(target * torch.log(pred + eps) +
                (1 - target) * torch.log(1 - pred + eps))

        return (focal_weight * bce).mean()


class HybridLoss(nn.Module):
    def __init__(self, lambda_focal_bce=0.3):
        super().__init__()
        self.lambda_fbce = lambda_focal_bce
        self.fbce = FocalBCELoss(alpha=0.75, gamma=2.0)
        self.ft = FocalTverskyLoss(alpha=0.2, beta=0.8, gamma=1.5, smooth=1.0)

    def forward(self, pred, target):
        target = target.to(pred.device)
        fbce_loss = self.fbce(pred, target)
        ft_loss = self.ft(pred, target)
        return self.lambda_fbce * fbce_loss + (1.0 - self.lambda_fbce) * ft_loss


loss_fn = HybridLoss(lambda_focal_bce=0.3).to(DEVICE)
print("Loss ready on", DEVICE)
 

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# SECTION 8 — METRICS
# Rule: accumulate TP/FP/FN/TN across ALL batches, compute once at the end.
# Never average per-batch (that gives different results for unequal batch sizes).
# ──────────────────────────────────────────────────────────────────────────────
 
def compute_metrics(pred_logits, true_mask, threshold=THRESHOLD):
    """Per-batch metric helper. Returns raw TP/FP/FN/TN counts + derived metrics."""
    pred   = (torch.sigmoid(pred_logits) > threshold).float().view(-1)
    target = true_mask.float().view(-1)
    eps    = 1e-8
 
    tp = ((pred == 1) & (target == 1)).sum().float()
    fp = ((pred == 1) & (target == 0)).sum().float()
    fn = ((pred == 0) & (target == 1)).sum().float()
    tn = ((pred == 0) & (target == 0)).sum().float()
 
    return {
        "tp": tp.item(), "fp": fp.item(), "fn": fn.item(), "tn": tn.item(),
        "iou":       (tp / (tp + fp + fn + eps)).item(),
        "precision": (tp / (tp + fp + eps)).item(),
        "recall":    (tp / (tp + fn + eps)).item(),
        "f1":        (2*tp / (2*tp + fp + fn + eps)).item(),
    }
 
 
def compute_epoch_metrics(tp, fp, fn, tn):
    """Compute metrics from accumulated totals (the correct way)."""
    eps = 1e-8
    tp, fp, fn, tn = [torch.tensor(x, dtype=torch.float32) for x in (tp, fp, fn, tn)]
    return {
        "iou":       (tp / (tp + fp + fn + eps)).item(),
        "precision": (tp / (tp + fp + eps)).item(),
        "recall":    (tp / (tp + fn + eps)).item(),
        "f1":        (2*tp / (2*tp + fp + fn + eps)).item(),
    }
 
 

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# SECTION 9 — EVALUATION FUNCTIONS
# ──────────────────────────────────────────────────────────────────────────────
 
def standard_evaluate(model, loader, loss_fn, device, threshold=THRESHOLD):
    """
    Simple single-pass evaluation.
    Faster than TTA but slightly less accurate.
    Good for quick iteration.
    """
    model.eval()
    total_loss, total_pixels = 0.0, 0
    tp = fp = fn = tn = 0.0
 
    with torch.no_grad():
        for eo, sar, masks, _ in loader:
            eo    = eo.to(device)
            sar   = sar.to(device)
            masks = masks.to(device)
 
            logits = model(eo, sar)           # (B, 1, H, W) raw logits
            loss   = loss_fn(logits, masks)
 
            n_px          = eo.size(0) * masks.shape[-2] * masks.shape[-1]
            total_loss   += loss.item() * n_px
            total_pixels += n_px
 
            m   = compute_metrics(logits, masks, threshold)
            tp += m["tp"]; fp += m["fp"]
            fn += m["fn"]; tn += m["tn"]
 
    return {
        "loss": total_loss / max(total_pixels, 1),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }
 
 
def tta_evaluate(model, loader, loss_fn, device, threshold=THRESHOLD):
    """
    4-flip Test-Time Augmentation:
      - original image
      - horizontal flip → predict → un-flip
      - vertical flip   → predict → un-flip
      - both flips      → predict → un-flip
    Average the 4 probability maps before thresholding.
 
    Why this helps: the model sees each image from 4 perspectives.
    Averaging reduces prediction variance, especially near boundaries.
    """
    model.eval()
    total_loss, total_pixels = 0.0, 0
    tp = fp = fn = tn = 0.0
 
    with torch.no_grad():
        for eo, sar, masks, _ in loader:
            eo    = eo.to(device)    # (B, 3, H, W)
            sar   = sar.to(device)   # (B, 6, H, W)
            masks = masks.to(device) # (B, H, W)
 
            # Helper: forward pass → sigmoid probability
            def fwd(eo_t, sar_t):
                return torch.sigmoid(model(eo_t, sar_t))  # (B, 1, H, W) probabilities
 
            # Original
            p0 = fwd(eo, sar)
 
            # Horizontal flip (dims=[3] = width axis)
            # Flip input → predict → flip output back to original orientation
            p1 = torch.flip(
                fwd(torch.flip(eo, [3]), torch.flip(sar, [3])),
                [3]
            )
 
            # Vertical flip (dims=[2] = height axis)
            p2 = torch.flip(
                fwd(torch.flip(eo, [2]), torch.flip(sar, [2])),
                [2]
            )
 
            # Both flips combined (180° rotation)
            p3 = torch.flip(
                fwd(torch.flip(eo, [2, 3]), torch.flip(sar, [2, 3])),
                [2, 3]
            )
 
            # Average all 4 probability maps
            avg_prob = (p0 + p1 + p2 + p3) / 4.0   # (B, 1, H, W)
 
            # Convert back to logit for loss computation
            # logit = log(p / (1-p))  — inverse of sigmoid
            avg_logit = torch.log(avg_prob / (1 - avg_prob + 1e-8) + 1e-8)
 
            loss = loss_fn(avg_logit, masks)
 
            n_px          = eo.size(0) * masks.shape[-2] * masks.shape[-1]
            total_loss   += loss.item() * n_px
            total_pixels += n_px
 
            m   = compute_metrics(avg_logit, masks, threshold)
            tp += m["tp"]; fp += m["fp"]
            fn += m["fn"]; tn += m["tn"]
 
    return {
        "loss": total_loss / max(total_pixels, 1),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }
 
 
def print_results(split_name, stats, checkpoint_name, threshold, mode="TTA"):
    """Pretty-print evaluation results."""
    metrics = compute_epoch_metrics(
        stats["tp"], stats["fp"], stats["fn"], stats["tn"]
    )
    print("=" * 65)
    print(f"{split_name} RESULTS — {checkpoint_name}")
    print(f"Mode: {mode} | Threshold: {threshold}")
    print("=" * 65)
    print(f"  Loss:      {stats['loss']:.4f}")
    print(f"  IoU:       {metrics['iou']:.4f}    ← primary metric")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1:        {metrics['f1']:.4f}")
    print(f"  TP: {stats['tp']:.0f} | FP: {stats['fp']:.0f} | "
          f"FN: {stats['fn']:.0f} | TN: {stats['tn']:.0f}")
    return metrics
 
 

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# SECTION 10 — LOAD MODEL + RUN EVALUATION
# ──────────────────────────────────────────────────────────────────────────────
 
print(f"\nLoading model from: {CHECKPOINT}")
best_model = DualEncoderUNet(IN_CH_EO, IN_CH_SAR, NUM_CLASSES).to(DEVICE)
best_model.load_state_dict(
    torch.load(CHECKPOINT, map_location=DEVICE, weights_only=True)
)
best_model.eval()
print("Model loaded ✓")
 
# ── Mode 1: Standard eval (fast) ──────────────────────────────────────────────
print("\n" + "─" * 65)
print("STANDARD EVALUATION (single-pass, no TTA)")
print("─" * 65)
 
val_stats_std  = standard_evaluate(best_model, val_loader, loss_fn, DEVICE, THRESHOLD)
test_stats_std = standard_evaluate(best_model, test_loader, loss_fn, DEVICE, THRESHOLD)
 
val_metrics_std  = print_results("VALIDATION", val_stats_std, CHECKPOINT, THRESHOLD, "Standard")
test_metrics_std = print_results("TEST",       test_stats_std, CHECKPOINT, THRESHOLD, "Standard")
 
# ── Mode 2: TTA eval (more accurate) ──────────────────────────────────────────
print("\n" + "─" * 65)
print("TTA EVALUATION (4-flip test-time augmentation)")
print("─" * 65)
 
val_stats_tta  = tta_evaluate(best_model, val_loader, loss_fn, DEVICE, THRESHOLD)
test_stats_tta = tta_evaluate(best_model, test_loader, loss_fn, DEVICE, THRESHOLD)
 
val_metrics_tta  = print_results("VALIDATION", val_stats_tta, CHECKPOINT, THRESHOLD, "TTA")
test_metrics_tta = print_results("TEST",       test_stats_tta, CHECKPOINT, THRESHOLD, "TTA")
 
# ── Mode 3: Threshold sweep on validation ─────────────────────────────────────
print("\n" + "─" * 65)
print("THRESHOLD SWEEP — Validation set (TTA)")
print("─" * 65)
print(f"{'thr':>5} | {'IoU':>7} | {'Precision':>9} | {'Recall':>7} | {'F1':>7}")
print("-" * 50)
 
best_sweep_iou = 0.0
best_sweep_thr = THRESHOLD
 
for thr in SWEEP_THRESHOLDS:
    stats   = tta_evaluate(best_model, val_loader, loss_fn, DEVICE, threshold=thr)
    metrics = compute_epoch_metrics(
        stats["tp"], stats["fp"], stats["fn"], stats["tn"]
    )
    marker = " ← best" if metrics["iou"] > best_sweep_iou else ""
    print(
        f"{thr:>5.2f} | {metrics['iou']:>7.4f} | "
        f"{metrics['precision']:>9.4f} | {metrics['recall']:>7.4f} | "
        f"{metrics['f1']:>7.4f}{marker}"
    )
    if metrics["iou"] > best_sweep_iou:
        best_sweep_iou = metrics["iou"]
        best_sweep_thr = thr
 
print(f"\nBest validation threshold: {best_sweep_thr:.2f} → IoU = {best_sweep_iou:.4f}")
 
# ── Final summary ──────────────────────────────────────────────────────────────
print("\n" + "═" * 65)
print("FINAL SUMMARY")
print("═" * 65)
print(f"Checkpoint:          {CHECKPOINT}")
print(f"Val  IoU (standard): {val_metrics_std['iou']:.4f}")
print(f"Val  IoU (TTA):      {val_metrics_tta['iou']:.4f}")
print(f"Test IoU (standard): {test_metrics_std['iou']:.4f}")
print(f"Test IoU (TTA):      {test_metrics_tta['iou']:.4f}")
print(f"Best threshold:      {best_sweep_thr:.2f} (Val IoU: {best_sweep_iou:.4f})")
print("═" * 65)